In [17]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.nn import Module, Linear, ReLU, Sequential, ModuleList
from torch_geometric.nn import TransformerConv, GatedGraphConv, LayerNorm
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.transforms import AddRandomWalkPE
from sklearn.preprocessing import MinMaxScaler
import matplotlib.pyplot as plt
import pickle
import os
import zipfile
import torch_geometric
import math

# Initialize CUDA if available
if torch.cuda.is_available():
    torch.cuda.init()
    print(f"CUDA initialized. Available GPUs: {torch.cuda.device_count()}")

# Print torch_geometric version
print(f"torch_geometric version: {torch_geometric.__version__}")

# Fix random seed for reproducibility
torch.manual_seed(42)
np.random.seed(42)

# Set device for multi-GPU or CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if torch.cuda.is_available() and torch.cuda.device_count() > 1:
    print(f"Using {torch.cuda.device_count()} GPUs!")
else:
    print(f"Using a single GPU or CPU.")

# Extract dataset if not already extracted
zip_path = 'METR-LA.zip'
data_dir = 'METR-LA'
if not os.path.exists(data_dir):
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(data_dir)
    print(f"Extracted dataset to '{data_dir}' directory.")

CUDA initialized. Available GPUs: 2
torch_geometric version: 2.6.1
Using 2 GPUs!


In [18]:
def load_adjacency_matrix(adj_matrix_path, data_dir='METR-LA'):
    try:
        with open(adj_matrix_path, 'rb') as f:
            adj_data = pickle.load(f)

        # Handle various formats found in pickle files (tuple, dict, list)
        if isinstance(adj_data, tuple):
            adj_matrix = next((np.array(item) for item in adj_data if isinstance(item, (np.ndarray, list)) and np.array(item).ndim == 2), None)
        elif isinstance(adj_data, dict):
            possible_keys = ['adj', 'adj_mx', 'matrix']
            key = next((k for k in possible_keys if k in adj_data), None)
            adj_matrix = adj_data.get(key)
        else:
            adj_matrix = np.array(adj_data)

        if adj_matrix is None or adj_matrix.ndim != 2 or adj_matrix.shape[0] != adj_matrix.shape[1]:
             raise ValueError("Could not extract a valid square 2D matrix.")

        adj_matrix = adj_matrix.astype(np.float32)
        print(f"Loaded adjacency matrix from {adj_matrix_path}, shape: {adj_matrix.shape}")
        return adj_matrix

    except Exception as e:
        print(f"Warning: Failed to load from {adj_matrix_path}: {e}. Falling back to default.")
        default_path = os.path.join(data_dir, 'adj_mat.npy')
        adj_matrix = np.load(default_path)
        print(f"Loaded default adjacency matrix, shape: {adj_matrix.shape}")
        return adj_matrix


def load_metr_la_data(data_dir='METR-LA', num_days=99, custom_adj_path=None):
    if custom_adj_path:
        print(f"Loading custom adjacency matrix from: {custom_adj_path}")
        adj_mat = load_adjacency_matrix(custom_adj_path, data_dir)
    else:
        default_path = os.path.join(data_dir, 'adj_mat.npy')
        adj_mat = np.load(default_path)
        print(f"Loaded default adjacency matrix from {default_path}")

    node_values = np.load(os.path.join(data_dir, 'node_values.npy'))
    time_steps = num_days * 288
    node_values = node_values[:time_steps]

    if node_values.ndim == 3:
        node_values = node_values[:, :, 0].T
    elif node_values.ndim != 2:
        raise ValueError(f"Expected 2D or 3D node_values, got shape {node_values.shape}")

    print(f"Loaded node data for {num_days} days, shape: {node_values.shape}")

    scaler = MinMaxScaler()
    node_values = scaler.fit_transform(node_values.T).T

    edge_index = torch.tensor(np.where(adj_mat > 0), dtype=torch.long)
    edge_weight = torch.tensor(adj_mat[edge_index[0], edge_index[1]], dtype=torch.float)

    edge_index = edge_index.to(device)
    edge_weight = edge_weight.to(device)

    return node_values, edge_index, edge_weight, scaler

In [19]:
def add_time_features(features, current_time_step_index, num_intervals_per_day=288, in_timestep=12):
    """Adds cyclical time-of-day features to the input features."""
    batch_size, current_in_timestep, num_nodes = features.shape

    # Generate time-of-day features for each time step within the input window
    time_features_list = []
    for i in range(current_in_timestep):
        global_time_step = (current_time_step_index + i) % num_intervals_per_day
        sin_time = math.sin(2 * math.pi * global_time_step / num_intervals_per_day)
        cos_time = math.cos(2 * math.pi * global_time_step / num_intervals_per_day)
        time_features = torch.tensor([[sin_time, cos_time]] * num_nodes, dtype=torch.float32)
        time_features_list.append(time_features)

    # Stack to get shape (in_timestep, num_nodes, 2)
    time_features_tensor = torch.stack(time_features_list, dim=0)
    # Repeat for each sample in the batch
    time_features_batch = time_features_tensor.unsqueeze(0).repeat(batch_size, 1, 1, 1)
    # Reshape original features and concatenate
    features_reshaped = features.unsqueeze(-1)
    features_with_time = torch.cat((features_reshaped, time_features_batch), dim=-1)

    return features_with_time


def create_data_samples(node_values, in_timestep=12, out_timestep=12, stride=1):
    """Creates input-output samples from the node values using a sliding window with a given stride."""
    features_list, labels_list = [], []
    num_nodes, total_time_steps = node_values.shape
    time_step_indices = range(0, total_time_steps - in_timestep - out_timestep + 1, stride)

    for i in time_step_indices:
        features_list.append(node_values[:, i:i + in_timestep])
        labels_list.append(node_values[:, i + in_timestep:i + in_timestep + out_timestep])

    features = torch.tensor(np.array(features_list, dtype=np.float32)).permute(0, 2, 1)
    labels = torch.tensor(np.array(labels_list, dtype=np.float32)).permute(0, 2, 1)

    features_with_time_list = [
        add_time_features(
            features[i].unsqueeze(0),
            global_start_time_step,
            in_timestep=in_timestep
        ).squeeze(0)
        for i, global_start_time_step in enumerate(time_step_indices)
    ]

    features_with_time = torch.stack(features_with_time_list, dim=0)
    features_final = features_with_time.permute(0, 2, 1, 3).reshape(features_with_time.shape[0], num_nodes, -1)
    return features_final, labels


def create_100th_day_data(data_dir='METR-LA', in_timestep=12, out_timestep=12, stride=1):
    """Creates samples for the 100th day with a given stride for evaluation."""
    node_values_full = np.load(os.path.join(data_dir, 'node_values.npy'))
    start_global_time_index = 288 * 99
    node_values_day100 = node_values_full[start_global_time_index : start_global_time_index + 288]
    node_values = node_values_day100[:, :, 0].T

    scaler_100 = MinMaxScaler()
    node_values_scaled = scaler_100.fit_transform(node_values.T).T

    # Reuse the sampling logic, passing the scaled 100th-day data
    # We must adjust the global time index for correct time feature creation
    features, labels = create_data_samples(node_values_scaled, in_timestep, out_timestep, stride)

    # Re-create time features with the correct global index offset
    num_nodes = node_values.shape[0]
    time_step_indices = range(0, node_values_scaled.shape[1] - in_timestep - out_timestep + 1, stride)
    speed_features = features.reshape(features.shape[0], num_nodes, in_timestep, -1)[:, :, :, 0].permute(0, 2, 1)

    features_with_time_list = [
        add_time_features(
            speed_features[i].unsqueeze(0),
            start_global_time_index + local_start_time_step,
            in_timestep=in_timestep
        ).squeeze(0)
        for i, local_start_time_step in enumerate(time_step_indices)
    ]
    
    features_with_time = torch.stack(features_with_time_list, dim=0)
    features_final = features_with_time.permute(0, 2, 1, 3).reshape(features_with_time.shape[0], num_nodes, -1)
    
    return features_final, labels, scaler_100

In [20]:
def compute_metrics(pred, target, scaler):
    batch_size, num_nodes, out_timestep = pred.shape
    pred_np = pred.cpu().numpy().transpose(0, 2, 1).reshape(-1, num_nodes)
    target_np = target.cpu().numpy().transpose(0, 2, 1).reshape(-1, num_nodes)
    pred_unscaled = scaler.inverse_transform(pred_np).reshape(batch_size, out_timestep, num_nodes).transpose(0, 2, 1)
    target_unscaled = scaler.inverse_transform(target_np).reshape(batch_size, out_timestep, num_nodes).transpose(0, 2, 1)
    pred_unscaled = torch.tensor(pred_unscaled, device=pred.device)
    target_unscaled = torch.tensor(target_unscaled, device=target.device)
    mae = torch.mean(torch.abs(pred_unscaled - target_unscaled)).item()
    rmse = torch.sqrt(torch.mean((pred_unscaled - target_unscaled) ** 2)).item()
    mask = target_unscaled > 1.0
    mape = torch.where(mask, torch.abs((pred_unscaled - target_unscaled) / (target_unscaled + 1e-2)), 0.0)
    mape = torch.mean(mape[mask]) * 100 if mask.sum() > 0 else 0.0
    return mae, rmse, mape


def visualize_predictions(pred, target, scaler, output_dir='plots', num_sensors=5, filename='prediction.png'):
    """Visualizes predictions for the first sample in a batch."""
    os.makedirs(output_dir, exist_ok=True)
    batch_size, num_nodes, out_timestep = pred.shape

    pred_unscaled = scaler.inverse_transform(pred.cpu().numpy()[0].T).T
    target_unscaled = scaler.inverse_transform(target.cpu().numpy()[0].T).T

    plt.figure(figsize=(12, 10))
    for i in range(min(num_sensors, num_nodes)):
        plt.subplot(num_sensors, 1, i + 1)
        plt.plot(pred_unscaled[i], label='Predicted', marker='o', markersize=4, linestyle='--')
        plt.plot(target_unscaled[i], label='Actual', marker='x', markersize=4, linestyle='-')
        plt.title(f'Sensor {i + 1} Traffic Speed Prediction (1-Hour Horizon)')
        plt.xlabel('Time Step (5-min intervals)')
        plt.ylabel('Speed (mph)')
        plt.legend()
        plt.grid(True)
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, filename))
    plt.close()


def visualize_full_100th_day(day_100_loader, model, scaler, output_dir='plots', num_sensors=5, filename='full_day.png'):
    """Reconstructs and visualizes the full 100th day prediction."""
    model.eval()
    all_preds_list, all_targets_list = [], []
    with torch.no_grad():
        for data in day_100_loader:
            data = data.to(device)
            pe = data.pe if hasattr(data, 'pe') else None
            out = model(data.x, data.edge_index, data.edge_attr, pe, data.batch)
            target = data.y.view(data.batch.max().item() + 1, out.shape[1], -1)
            all_preds_list.append(out.cpu())
            all_targets_list.append(target.cpu())

    # Concatenate list of batch tensors into a single numpy array of samples
    all_preds = torch.cat(all_preds_list, axis=0).numpy()
    all_targets = torch.cat(all_targets_list, axis=0).numpy()

    num_samples, num_nodes, out_timestep = all_preds.shape
    in_timestep = 12

    # Accumulator arrays for averaging overlapping predictions
    full_day_preds_scaled = np.zeros((num_nodes, 288))
    full_day_targets_scaled = np.zeros((num_nodes, 288))
    counts = np.zeros((num_nodes, 288))

    for i in range(num_samples):
        start_idx = i + in_timestep
        end_idx = start_idx + out_timestep
        if end_idx > 288:
            continue

        # Accumulate the first step of each prediction window for both preds and targets
        full_day_preds_scaled[:, start_idx] += all_preds[i, :, 0]
        full_day_targets_scaled[:, start_idx] += all_targets[i, :, 0]
        counts[:, start_idx] += 1

    # Average the values in overlapping windows
    predicted_indices = np.where(counts > 0)
    full_day_preds_scaled[predicted_indices] /= counts[predicted_indices]
    full_day_targets_scaled[predicted_indices] /= counts[predicted_indices]

    # Unscale the entire reconstructed day series at once
    # Note: The scaler expects shape (n_samples, n_features), so we transpose, unscale, and transpose back.
    full_day_preds_unscaled = scaler.inverse_transform(full_day_preds_scaled.T).T
    full_day_targets_unscaled = scaler.inverse_transform(full_day_targets_scaled.T).T

    # Plotting
    plot_indices = np.where(counts[0] > 0)[0]
    time_axis = np.arange(288) * 5 / 60  # Time in hours
    plt.figure(figsize=(15, 12))
    for i in range(min(num_sensors, num_nodes)):
        plt.subplot(num_sensors, 1, i + 1)
        plt.plot(time_axis[plot_indices], full_day_preds_unscaled[i, plot_indices], label='Predicted', alpha=0.9)
        plt.plot(time_axis[plot_indices], full_day_targets_unscaled[i, plot_indices], label='Actual', alpha=0.7, linestyle='--')
        plt.title(f'Sensor {i + 1} - Full 100th Day Traffic Speed')
        plt.xlabel('Time (Hour of Day)')
        plt.ylabel('Speed (mph)')
        plt.legend()
        plt.grid(True)
        plt.xlim(0, 24)
    plt.tight_layout()
    os.makedirs(output_dir, exist_ok=True)
    plt.savefig(os.path.join(output_dir, filename))
    plt.close()

In [21]:
class GPSLayer(nn.Module):
    """A single layer of the GPS model, combining GCN and Transformer."""
    def __init__(self, hidden_channels, heads=4, dropout=0.1):
        super().__init__()
        self.gated_gcn = GatedGraphConv(hidden_channels, num_layers=1)
        self.transformer = TransformerConv(
            hidden_channels, hidden_channels, heads=heads, concat=False, dropout=dropout
        )
        self.norm1 = LayerNorm(hidden_channels)
        self.norm2 = LayerNorm(hidden_channels)
        self.ffn = nn.Sequential(
            nn.Linear(hidden_channels, hidden_channels * 2),
            nn.ReLU(),
            nn.Linear(hidden_channels * 2, hidden_channels),
            nn.Dropout(dropout)
        )
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x, edge_index, edge_weight, batch):
        # Local message passing (GCN)
        x_gcn = self.gated_gcn(x, edge_index, edge_weight=edge_weight)
        
        # Global attention (Transformer)
        x_trans = self.transformer(x, edge_index)
        
        # Combine and apply residual connections
        x = x + self.dropout(x_gcn) + self.dropout(x_trans)
        x = self.norm1(x)
        
        # Feed-forward network
        x = x + self.ffn(x)
        x = self.norm2(x)
        return x


class GPSGraphTransformer(nn.Module):
    """The main Graph GPS model for traffic forecasting."""
    def __init__(self, in_channels, hidden_channels=64, out_channels=12, num_layers=4, pe_dim=20, heads=4, use_pe=True):
        super().__init__()
        self.use_pe = use_pe
        
        # Embedding layer for input features and positional encodings
        self.node_emb = nn.Linear(in_channels, hidden_channels - pe_dim if use_pe else hidden_channels)
        if use_pe:
            self.pe_lin = nn.Linear(pe_dim, pe_dim)
            
        self.layers = nn.ModuleList(
            [GPSLayer(hidden_channels, heads=heads) for _ in range(num_layers)]
        )
        self.norm = LayerNorm(hidden_channels)
        
        # Output MLP to produce the forecast
        self.mlp = nn.Sequential(
            nn.Linear(hidden_channels, hidden_channels // 2),
            nn.ReLU(),
            nn.Linear(hidden_channels // 2, out_channels)
        )
    
    def forward(self, x, edge_index, edge_weight, pe=None, batch=None):
        # Initial node embedding
        x = self.node_emb(x)
        if self.use_pe and pe is not None:
            x = torch.cat([x, self.pe_lin(pe)], dim=-1)
            
        # Process through GPS layers
        for layer in self.layers:
            x = layer(x, edge_index, edge_weight, batch)
            
        x = self.norm(x)
        x = self.mlp(x)
        
        # Reshape to (batch_size, num_nodes, out_channels)
        batch_size = batch.max().item() + 1 if batch is not None else 1
        num_nodes = x.size(0) // batch_size
        return x.view(batch_size, num_nodes, -1)

In [22]:
# --- User Input Section (Modified for iteration) ---
adj_matrices = [
    'adj_mx_knn_lane.pkl', 'adj_mx_knn.pkl',
    'adj_mx_radius_lane.pkl', 'adj_mx_radius.pkl'
]
stride_options = {'1': ('5 minutes', 1), '2': ('1 hour', 12), '3': ('1 day', 288)}

results = {} # To store results for all runs

for adj_file in adj_matrices:
    for stride_key, (stride_name, stride_val) in stride_options.items():
        print(f"\n--- Processing Adjacency Matrix: {adj_file}, Stride: {stride_name} ({stride_val} time steps) ---")
        
        try:
            node_values, edge_index, edge_weight, scaler = load_metr_la_data(custom_adj_path=f'METR-LA/{adj_file}')
            in_timestep, out_timestep = 12, 12

            features, labels = create_data_samples(node_values, in_timestep, out_timestep, stride=stride_val)

            total_samples = len(features)
            train_end = int(0.6 * total_samples)
            val_end = train_end + int(0.2 * total_samples)

            train_features, train_labels = features[:train_end], labels[:train_end]
            val_features, val_labels = features[train_end:val_end], labels[train_end:val_end]
            test_features, test_labels = features[val_end:], labels[val_end:]
            print(f"Total Samples: {total_samples} -> Train: {len(train_features)}, Val: {len(val_features)}, Test: {len(test_features)}")

            transform = AddRandomWalkPE(walk_length=20, attr_name='pe')
            def create_pyg_dataset(features, labels):
                # Ensure edge_index and edge_weight are passed as arguments to Data
                return [transform(Data(x=f, y=l.T, edge_index=edge_index, edge_attr=edge_weight)) for f, l in zip(features, labels)]


            train_data = create_pyg_dataset(train_features, train_labels)
            val_data = create_pyg_dataset(val_features, val_labels)
            test_data = create_pyg_dataset(test_features, test_labels)
            
            day_100_features, day_100_labels, day_100_scaler = create_100th_day_data(in_timestep=in_timestep, out_timestep=out_timestep, stride=1)
            day_100_data = create_pyg_dataset(day_100_features, day_100_labels)

            train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
            val_loader = DataLoader(val_data, batch_size=32, shuffle=False)
            test_loader = DataLoader(test_data, batch_size=32, shuffle=False)
            day_100_loader = DataLoader(day_100_data, batch_size=32, shuffle=False)

            model = GPSGraphTransformer(in_channels=in_timestep * 3, hidden_channels=64, out_channels=out_timestep).to(device)
            optimizer = optim.AdamW(model.parameters(), lr=0.001, weight_decay=0.01)
            criterion = nn.MSELoss()

            print("\n--- Starting Training ---")
            for epoch in range(20): # Using 20 epochs for demonstration
                model.train()
                total_train_loss = 0
                for data in train_loader:
                    data = data.to(device)
                    optimizer.zero_grad()
                    out = model(data.x, data.edge_index, data.edge_attr, data.pe, data.batch)
                    target = data.y.view(data.batch.max().item() + 1, out.shape[1], -1)
                    loss = criterion(out, target)
                    loss.backward()
                    optimizer.step()
                    total_train_loss += loss.item()

                model.eval()
                with torch.no_grad():
                    # Corrected validation loss calculation to handle empty val_loader
                    total_val_loss = 0
                    if len(val_loader) > 0:
                        for d in val_loader:
                            d = d.to(device)
                            out = model(d.x, d.edge_index, d.edge_attr, d.pe, d.batch)
                            target = d.y.view(d.batch.max().item() + 1, out.shape[1], -1)
                            total_val_loss += criterion(out, target).item()
                        total_val_loss /= len(val_loader)
                    else:
                        total_val_loss = float('nan') # Or handle as appropriate if no validation data
                
                print(f"Epoch {epoch + 1:02d}, Train Loss: {total_train_loss / len(train_loader):.4f}, Val Loss: {total_val_loss:.4f}")

            print("\n--- Evaluating Model ---")
            def evaluate(loader, scaler_obj, set_name):
                model.eval()
                total_mae, total_rmse, total_mape, count = 0, 0, 0, 0
                with torch.no_grad():
                    for i, data in enumerate(loader):
                        data = data.to(device)
                        out = model(data.x, data.edge_index, data.edge_attr, data.pe, data.batch)
                        target = data.y.view(data.batch.max().item() + 1, out.shape[1], -1)
                        mae, rmse, mape = compute_metrics(out, target, scaler_obj)
                        total_mae += mae * len(data)
                        total_rmse += rmse * len(data)
                        total_mape += mape * len(data)
                        count += len(data)
                        if i == 0: visualize_predictions(out, target, scaler_obj, f'plots_{adj_file.split(".")[0]}_{stride_val}', filename=f'preds_{adj_file.split(".")[0]}_{stride_val}.png')
                
                # Avoid division by zero if count is 0
                if count > 0:
                    print(f"{set_name} Results -> MAE: {total_mae/count:.4f}, RMSE: {total_rmse/count:.4f}, MAPE: {total_mape/count:.4f}%")
                    return {'mae': total_mae/count, 'rmse': total_rmse/count, 'mape': total_mape/count}
                else:
                    print(f"{set_name} Results -> No data to evaluate.")
                    return {'mae': float('nan'), 'rmse': float('nan'), 'mape': float('nan')}

            test_metrics = evaluate(test_loader, scaler, "Test")
            day_100_metrics = evaluate(day_100_loader, day_100_scaler, "100th Day")
            
            visualize_full_100th_day(day_100_loader, model, day_100_scaler, output_dir=f'plots_{adj_file.split(".")[0]}_{stride_val}', filename=f'full_day_{adj_file.split(".")[0]}_{stride_val}.png')
            print("Full 100th day visualization saved.")
            
            results[f"{adj_file}_stride_{stride_val}"] = {'test': test_metrics, '100th_day': day_100_metrics}

        except Exception as e:
            print(f"An error occurred for {adj_file} with stride {stride_val}: {e}")
            results[f"{adj_file}_stride_{stride_val}"] = {'test': {'mae': float('nan'), 'rmse': float('nan'), 'mape': float('nan')},
                                                           '100th_day': {'mae': float('nan'), 'rmse': float('nan'), 'mape': float('nan')}}

# --- Final Summary ---
print("\n--- Summary of All Results ---")
for key, metrics_dict in results.items():
    adj_file_name, stride_info = key.split('_stride_')
    print(f"\nAdjacency Matrix: {adj_file_name}, Stride: {stride_info}")
    print(f"  Test      - MAE: {metrics_dict['test']['mae']:.4f}, RMSE: {metrics_dict['test']['rmse']:.4f}, MAPE: {metrics_dict['test']['mape']:.4f}%")
    print(f"  100th Day - MAE: {metrics_dict['100th_day']['mae']:.4f}, RMSE: {metrics_dict['100th_day']['rmse']:.4f}, MAPE: {metrics_dict['100th_day']['mape']:.4f}%")


--- Processing Adjacency Matrix: adj_mx_knn_lane.pkl, Stride: 5 minutes (1 time steps) ---
Loading custom adjacency matrix from: METR-LA/adj_mx_knn_lane.pkl
Loaded adjacency matrix from METR-LA/adj_mx_knn_lane.pkl, shape: (207, 207)
Loaded node data for 99 days, shape: (207, 28512)
Total Samples: 28489 -> Train: 17093, Val: 5697, Test: 5699

--- Starting Training ---
Epoch 01, Train Loss: 0.0335, Val Loss: 0.0242
Epoch 02, Train Loss: 0.0258, Val Loss: 0.0229
Epoch 03, Train Loss: 0.0249, Val Loss: 0.0227
Epoch 04, Train Loss: 0.0244, Val Loss: 0.0213
Epoch 05, Train Loss: 0.0240, Val Loss: 0.0208
Epoch 06, Train Loss: 0.0238, Val Loss: 0.0207
Epoch 07, Train Loss: 0.0237, Val Loss: 0.0198
Epoch 08, Train Loss: 0.0235, Val Loss: 0.0198
Epoch 09, Train Loss: 0.0234, Val Loss: 0.0225
Epoch 10, Train Loss: 0.0233, Val Loss: 0.0197
Epoch 11, Train Loss: 0.0231, Val Loss: 0.0195
Epoch 12, Train Loss: 0.0230, Val Loss: 0.0205
Epoch 13, Train Loss: 0.0228, Val Loss: 0.0193
Epoch 14, Train Lo